# ClearRisk Lab Quickstart: CCP Waterfall Baseline vs Stressed

This notebook runs a full **end-to-end** quickstart using the CLI report bundle path:

1. Build a baseline and stressed synthetic config.
2. Run `clearrisk report --compare-config ... --bundle-dir ...`.
3. Inspect generated reproducible artifacts in `reports/`.

Educational/research use only. This is not a production CCP model.

## 1) Environment Setup

The code below:
- locates repo paths,
- ensures `src/` is importable,
- creates a notebook-local output folder.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"
REPORTS_DIR = REPO_ROOT / "reports"
CONFIG_DIR = REPORTS_DIR / "quickstart_configs"
BUNDLE_DIR = REPORTS_DIR / "quickstart_bundle"

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repo root:", REPO_ROOT)
print("Bundle dir:", BUNDLE_DIR)

## 2) Create Baseline and Stressed Synthetic Configs

Baseline: moderate volatility, no liquidity close-out add-on.

Stressed: higher volatility and enabled close-out cost model.

In [ ]:
baseline_config = {
    "run_name": "quickstart_baseline",
    "time_horizon_days": 252,
    "include_liquidity_closeout_cost": False,
    "margin": {
        "method": "gaussian_var",
        "confidence_level": 0.99,
        "margin_period_of_risk_days": 5,
        "volatility_floor": 0.10,
        "anti_procyclicality_buffer": 0.00,
        "concentration_addon": 0.02,
        "liquidity_addon": 0.02,
    },
    "waterfall": {
        "ccp_skin_in_game": 100000000,
        "assessment_cap_multiple": 0.20,
        "include_defaulter_df": True,
        "default_fund_method": "cover2",
        "default_fund_value": 0.0,
    },
    "closeout": {
        "base_spread_cost": 0.00,
        "volatility_liquidity_multiplier": 0.00,
        "concentration_penalty": 0.00,
        "market_depth_penalty": 0.00,
    },
    "contagion": {
        "liquidity_breach_ratio": 1.00,
    },
    "scenarios": [
        {
            "scenario_name": "gaussian_baseline",
            "volatility": 0.20,
            "liquidity_spread_multiplier": 1.00,
            "stress_label": "baseline",
        }
    ],
    "members": [
        {
            "member_id": "CM_A",
            "default_fund_contribution": 120000,
            "liquidity_buffer": 300000,
            "portfolio": {"positions": [{"asset_id": "IDX", "quantity": 1200000, "price": 1.0}]},
        },
        {
            "member_id": "CM_B",
            "default_fund_contribution": 100000,
            "liquidity_buffer": 250000,
            "portfolio": {"positions": [{"asset_id": "IDX", "quantity": 900000, "price": 1.0}]},
        },
        {
            "member_id": "CM_C",
            "default_fund_contribution": 80000,
            "liquidity_buffer": 200000,
            "portfolio": {"positions": [{"asset_id": "IDX", "quantity": 700000, "price": 1.0}]},
        },
    ],
}

stressed_config = {
    **baseline_config,
    "run_name": "quickstart_stressed",
    "include_liquidity_closeout_cost": True,
    "margin": {
        **baseline_config["margin"],
        "method": "stressed_var",
        "anti_procyclicality_buffer": 0.05,
        "liquidity_addon": 0.08,
    },
    "closeout": {
        "base_spread_cost": 0.01,
        "volatility_liquidity_multiplier": 0.02,
        "concentration_penalty": 0.06,
        "market_depth_penalty": 0.01,
    },
    "scenarios": [
        {
            "scenario_name": "high_vol_stress",
            "volatility": 0.40,
            "liquidity_spread_multiplier": 1.50,
            "stress_label": "stressed",
        }
    ],
}

baseline_path = CONFIG_DIR / "quickstart_baseline.json"
stressed_path = CONFIG_DIR / "quickstart_stressed.json"
baseline_path.write_text(json.dumps(baseline_config, indent=2), encoding="utf-8")
stressed_path.write_text(json.dumps(stressed_config, indent=2), encoding="utf-8")

print("Wrote:", baseline_path)
print("Wrote:", stressed_path)

## 3) Run CLI Report Bundle (Markdown + CSV + JSON)

This executes the same CLI flow you would run from terminal, and writes reproducible artifacts to:

- `reports/quickstart_bundle/report.md`
- `reports/quickstart_bundle/scenario_comparison.csv`
- `reports/quickstart_bundle/scenario_comparison_summary.json`

In [ ]:
env = dict(os.environ)
env["PYTHONPATH"] = f"{SRC_DIR}:{env.get('PYTHONPATH', '')}" if env.get("PYTHONPATH") else str(SRC_DIR)

cmd = [
    sys.executable,
    "-m",
    "clearrisk.cli",
    "report",
    "--config",
    str(stressed_path),
    "--compare-config",
    str(baseline_path),
    "--bundle-dir",
    str(BUNDLE_DIR),
]

completed = subprocess.run(cmd, env=env, check=True, capture_output=True, text=True)
print(completed.stdout)

## 4) Inspect Generated Artifacts

This cell reads the JSON + CSV bundle files and prints a compact view.

In [ ]:
import csv

report_path = BUNDLE_DIR / "report.md"
csv_path = BUNDLE_DIR / "scenario_comparison.csv"
summary_path = BUNDLE_DIR / "scenario_comparison_summary.json"

print("Exists report:", report_path.exists())
print("Exists csv:", csv_path.exists())
print("Exists json:", summary_path.exists())

summary = json.loads(summary_path.read_text(encoding="utf-8"))
print("\nComparison keys:", list(summary.get("comparison", {}).keys()))
print("Tail loss ratio (ES):", summary["comparison"].get("tail_loss_ratio_es"))
print("Margin adequacy delta:", summary["comparison"].get("margin_adequacy_delta"))
print("Cover-2 adequacy delta:", summary["comparison"].get("cover2_adequacy_delta"))

with csv_path.open("r", encoding="utf-8", newline="") as fh:
    reader = csv.DictReader(fh)
    rows = list(reader)

print("\nCSV rows (first 5):")
for row in rows[:5]:
    print(row)

In [ ]:
report_preview = report_path.read_text(encoding="utf-8").splitlines()
print("\n".join(report_preview[:40]))

## 5) Next Steps

- Move this quickstart into CI smoke checks for reproducible artifact generation.
- Add the Phase 4B/4C notebooks for tail-risk and Cover-2 + liquidity stress labs.
- Re-run with alternative margin methods (Gaussian VaR vs ES vs stressed VaR).